# Model-Based RL: Learned World Models

## 📚 Learning Objectives

By completing this notebook, you will:
- Understand model-based RL
- Learn world models from experience
- Use learned models for planning
- Compare model-based vs model-free

## 🔗 Prerequisites

- ✅ Understanding of RL fundamentals
- ✅ Understanding of neural networks
- ✅ Python, NumPy knowledge

---

## Official Structure Reference

This notebook covers practical activities from **Course 09, Unit 5**:
- Building model-based RL systems with learned world models

---

## Introduction

**Model-based RL** learns a model of the environment dynamics and uses it for planning, potentially achieving higher sample efficiency than model-free methods.

## Lesson Brief

This lesson introduces model-based RL, where the agent tries to learn how the environment works and then plan with that knowledge.

Students will see the full idea in simple form: collect experience, build a model, then use the model to choose actions.

Why this matters: model-based RL explains why some agents can learn faster from fewer environment interactions.

It also prepares students to think critically about model errors.

## Inputs and Outputs

**Inputs:** What we use in this notebook

- Libraries and concepts as introduced in this notebook; see prerequisites and code comments.

**Outputs:** What you'll see when you run the cells

- Printed results, figures, and summaries as shown when you run the cells.

---


In [1]:
import numpy as np

print("✅ Libraries imported!")
print("\nModel-Based RL: Learned World Models")
print("=" * 60)

print("\nModel-Based RL:")
print("  - Learn: Transition model P(s'|s,a) and reward model R(s,a)")
print("  - Plan: Use model to simulate and plan")
print("  - Act: Execute planned actions")

print("\nWorld Model Learning:")
print("  - Supervised learning from (s,a,s',r) tuples")
print("  - Neural networks to approximate dynamics")
print("  - Predictive models for next states/rewards")

print("\nBenefits:")
print("  - Higher sample efficiency")
print("  - Planning capabilities")
print("  - Better exploration")

print("\n✅ Model-based RL concepts understood!")

✅ Libraries imported!

Model-Based RL: Learned World Models

Model-Based RL:
  - Learn: Transition model P(s'|s,a) and reward model R(s,a)
  - Plan: Use model to simulate and plan
  - Act: Execute planned actions

World Model Learning:
  - Supervised learning from (s,a,s',r) tuples
  - Neural networks to approximate dynamics
  - Predictive models for next states/rewards

Benefits:
  - Higher sample efficiency
  - Planning capabilities
  - Better exploration

✅ Model-based RL concepts understood!


## Worked Example: Learn a Small World Model, Then Plan

In model-based RL, the agent tries to answer this question:

`If I take action a in state s, what will probably happen next?`

Below, the agent explores a tiny 1D environment, collects transitions, builds a
simple empirical model, and then uses that model to plan a path to the goal.

This is not a full modern world-model system, but it captures the core idea:
**experience -> model -> planning**.

In [2]:
import numpy as np

np.random.seed(42)
N_STATES = 6
GOAL = N_STATES - 1
ACTIONS = ["left", "right"]


def true_step(state, action):
    if action == "right":
        next_state = min(state + 1, GOAL)
    else:
        next_state = max(state - 1, 0)
    reward = 1.0 if next_state == GOAL else -0.1
    return next_state, reward


# Collect experience from random exploration.
transition_counts = {}
reward_sums = {}
for _ in range(200):
    state = np.random.randint(0, N_STATES)
    action = np.random.choice(ACTIONS)
    next_state, reward = true_step(state, action)
    transition_counts[(state, action, next_state)] = transition_counts.get((state, action, next_state), 0) + 1
    reward_sums[(state, action)] = reward_sums.get((state, action), 0.0) + reward


# Build a simple empirical model.
model_next_state = {}
model_reward = {}
for state in range(N_STATES):
    for action in ACTIONS:
        candidates = {
            ns: count
            for (s, a, ns), count in transition_counts.items()
            if s == state and a == action
        }
        if not candidates:
            continue
        best_next = max(candidates, key=candidates.get)
        total_visits = sum(candidates.values())
        model_next_state[(state, action)] = best_next
        model_reward[(state, action)] = reward_sums[(state, action)] / total_visits


# Plan with the learned model by preferring actions that move closer to the goal.
def plan_with_model(start_state):
    state = start_state
    trajectory = [state]
    chosen_actions = []
    total_reward = 0.0
    while state != GOAL:
        candidates = []
        for action in ACTIONS:
            next_state = model_next_state.get((state, action), state)
            predicted_reward = model_reward.get((state, action), -999)
            distance_to_goal = abs(GOAL - next_state)
            candidates.append((distance_to_goal, -predicted_reward, action, next_state))
        _, _, action, next_state = min(candidates)
        chosen_actions.append(action)
        total_reward += model_reward[(state, action)]
        state = next_state
        trajectory.append(state)
        if len(trajectory) > 10:
            break
    return trajectory, chosen_actions, total_reward


trajectory, chosen_actions, predicted_return = plan_with_model(start_state=0)

print("Learned model examples:")
for key in sorted(model_next_state)[:6]:
    print(f"  {key} -> next_state={model_next_state[key]}, reward={model_reward[key]:.2f}")

print("\nPlanned actions from state 0:", chosen_actions)
print("Planned trajectory from state 0:", trajectory)
print("Predicted return along planned path:", round(predicted_return, 2))

print("\nTeaching point:")
print("The agent first learns an approximate model of transitions and rewards, then")
print("uses that learned model to choose a sequence of actions that should move toward the goal.")

Learned model examples:
  (0, 'left') -> next_state=0, reward=-0.10
  (0, 'right') -> next_state=1, reward=-0.10
  (1, 'left') -> next_state=0, reward=-0.10
  (1, 'right') -> next_state=2, reward=-0.10
  (2, 'left') -> next_state=1, reward=-0.10
  (2, 'right') -> next_state=3, reward=-0.10

Planned actions from state 0: ['right', 'right', 'right', 'right', 'right']
Planned trajectory from state 0: [0, 1, 2, 3, 4, 5]
Predicted return along planned path: 0.6

Teaching point:
The agent first learns an approximate model of transitions and rewards, then
uses that learned model to choose a sequence of actions that should move toward the goal.


## 📚 References & Further Reading

**Books:**
- Sutton & Barto — [Reinforcement Learning: An Introduction](http://incompleteideas.net/book/the-book-2nd.html) (free online, the RL bible)

**Papers:**
- Mnih et al. (2015) — [Human-level control through deep RL (DQN)](https://www.nature.com/articles/nature14236)
- Silver et al. (2016) — [AlphaGo](https://www.nature.com/articles/nature16961)

**State-of-the-Art:** OpenAI Five beat world champions in Dota2; AlphaFold uses RL-like optimization for protein folding.

## Summary

This notebook focused on the core loop of model-based RL:

- collect experience from the environment
- learn an approximate transition and reward model
- plan actions using that learned model

### Main takeaway

Model-based RL can be attractive when environment interaction is expensive,
because the agent can reuse its model for planning instead of learning only from
real experience.

### Limitation to remember

If the learned model is wrong, planning can become confidently wrong too.
That is one of the main risks of model-based approaches.

## Closing Takeaway

**Teaching takeaway:** Model-based RL learns or uses environment dynamics so the agent can simulate consequences before acting.

**If students remember one idea:** Planning becomes possible when the agent has a workable model of how actions change the world.

**Quick check before moving on:**
- Can you explain the difference between acting from experience only and acting with a learned model?
- Can you describe one advantage and one risk of relying on a world model?

**Bridge to the next step:** This makes it easier to compare model-based and model-free RL directly in the next lesson.
